# Phase 3: Layout & OCR Extraction


In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr poppler-utils

!pip install pymupdf pytesseract pdf2image huggingface_hub pandas tqdm datasets

In [2]:
import os
import fitz  # PyMuPDF
import pytesseract
from pdf2image import convert_from_path
import pandas as pd
import json
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

print("Downloading dataset files from Hugging Face...")
dataset_path = snapshot_download(repo_id="BassemRamdan/data", repo_type="dataset")

categories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d)) and not d.startswith('.git')]

C:\Users\Delta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 2485 files:   0%|          | 0/2485 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
C:\Users\Delta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Delta\.cache\huggingface\hub\datasets--BassemRamdan--data. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In

In [ ]:
def extract_layout_pymupdf(pdf_path):
    """Extracts layout using PyMuPDF. Returns a list of pages with words and bboxes."""
    doc = fitz.open(pdf_path)
    pages_data = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        rect = page.rect
        page_width, page_height = rect.width, rect.height

        words = page.get_text("words")

        page_words = []
        for w in words:
            x0, y0, x1, y1, text = w[0], w[1], w[2], w[3], w[4]
            page_words.append({
                "text": text,
                "bbox": [float(x0), float(y0), float(x1), float(y1)]
            })

        pages_data.append({
            "page_num": page_num + 1,
            "width": float(page_width),
            "height": float(page_height),
            "words": page_words
        })

    doc.close()
    return pages_data

def extract_layout_tesseract(pdf_path, page_num_to_ocr=None):
    """Fallback: Renders PDF to image and extracts bounding boxes using Tesseract."""
    if page_num_to_ocr:
        images = convert_from_path(pdf_path, first_page=page_num_to_ocr, last_page=page_num_to_ocr)
    else:
        images = convert_from_path(pdf_path)

    pages_data = []

    for i, img in enumerate(images):
        width, height = img.size
        ocr_df = pytesseract.image_to_data(img, output_type=pytesseract.Output.DATAFRAME)

        ocr_df = ocr_df.dropna(subset=['text'])
        ocr_df = ocr_df[ocr_df['text'].str.strip() != '']

        page_words = []
        for _, row in ocr_df.iterrows():
            x0 = row['left']
            y0 = row['top']
            x1 = x0 + row['width']
            y1 = y0 + row['height']
            page_words.append({
                "text": str(row['text']),
                "bbox": [float(x0), float(y0), float(x1), float(y1)]
            })

        pages_data.append({
            "page_num": page_num_to_ocr if page_num_to_ocr else i + 1,
            "width": float(width),
            "height": float(height),
            "words": page_words
        })

    return pages_data

In [4]:
print("Starting Full Layout Extraction Pipeline...")
structured_resumes = []
ocr_used_count = 0

for category in tqdm(categories, desc="Categories"):
    cat_path = os.path.join(dataset_path, category)
    for filename in os.listdir(cat_path):
        if filename.lower().endswith('.pdf'):
            file_path = os.path.join(cat_path, filename)

            try:
                pages_data = extract_layout_pymupdf(file_path)

                for page in pages_data:
                    if len(page['words']) == 0:
                        print(f"\nNo text found natively on {category}/{filename} Page {page['page_num']}. Using Tesseract OCR...")
                        ocr_pages = extract_layout_tesseract(file_path, page_num_to_ocr=page['page_num'])
                        if ocr_pages:
                            page['words'] = ocr_pages[0]['words']
                            page['width'] = ocr_pages[0]['width']
                            page['height'] = ocr_pages[0]['height']
                        ocr_used_count += 1

                structured_resumes.append({
                    "filename": filename,
                    "category": category,
                    "pages": pages_data
                })

            except Exception as e:
                print(f"\nError processing {file_path}: {e}")

print(f"\nExtraction Complete. Tesseract Fallback was used {ocr_used_count} times.")

Starting Full Layout Extraction Pipeline...


Categories:   0%|          | 0/24 [00:00<?, ?it/s]


No text found natively on ACCOUNTANT/30304575.pdf Page 3. Using Tesseract OCR...

Error processing C:\Users\Delta\.cache\huggingface\hub\datasets--BassemRamdan--data\snapshots\2b8c61feeb3d3fbb54fe258d5a19516f110e29f8\ACCOUNTANT\30304575.pdf: Unable to get page count. Is poppler installed and in PATH?


Categories:  12%|█▎        | 3/24 [00:13<01:31,  4.35s/it]


No text found natively on APPAREL/56151548.pdf Page 2. Using Tesseract OCR...

Error processing C:\Users\Delta\.cache\huggingface\hub\datasets--BassemRamdan--data\snapshots\2b8c61feeb3d3fbb54fe258d5a19516f110e29f8\APPAREL\56151548.pdf: Unable to get page count. Is poppler installed and in PATH?


...


No text found natively on SALES/40987524.pdf Page 3. Using Tesseract OCR...

Error processing C:\Users\Delta\.cache\huggingface\hub\datasets--BassemRamdan--data\snapshots\2b8c61feeb3d3fbb54fe258d5a19516f110e29f8\SALES\40987524.pdf: Unable to get page count. Is poppler installed and in PATH?


Categories:  96%|█████████▌| 23/24 [01:41<00:04,  4.71s/it]


No text found natively on TEACHER/29797594.pdf Page 2. Using Tesseract OCR...

Error processing C:\Users\Delta\.cache\huggingface\hub\datasets--BassemRamdan--data\snapshots\2b8c61feeb3d3fbb54fe258d5a19516f110e29f8\TEACHER\29797594.pdf: Unable to get page count. Is poppler installed and in PATH?


Categories: 100%|██████████| 24/24 [01:44<00:00,  4.36s/it]


Extraction Complete. Tesseract Fallback was used 0 times.


In [5]:
output_file = "resume_layout_dataset.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(structured_resumes, f, ensure_ascii=False)

print(f"Saved layout data for {len(structured_resumes)} resumes to {output_file}.")
print("Size of JSON file on disk (MB):", os.path.getsize(output_file) / (1024 * 1024))



Saved layout data for 2466 resumes to resume_layout_dataset.json.
Size of JSON file on disk (MB): 202.74746704101562
